# 03 — Model Ablation Study

**Rift: Graph ML for Fraud Detection, Replay, and Audit**

This notebook trains and compares all four model variants:
1. **Baseline A**: Tabular XGBoost (features only)
2. **Baseline B**: GraphSAGE only (graph only)
3. **Flagship**: GraphSAGE + XGBoost hybrid
4. **Alternative**: GAT + XGBoost hybrid

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/03_model_ablation.ipynb)


## Environment Setup


In [ ]:
# Clone repo and install dependencies (run once)
import subprocess, sys, os

REPO = "https://github.com/AngelP17/Rift.git"
if not os.path.exists("/content/Rift"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)

os.chdir("/content/Rift")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "polars", "numpy", "pandas", "scikit-learn", "xgboost", "duckdb",
    "shap", "structlog", "python-dotenv", "rich", "jinja2", "pyarrow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
sys.path.insert(0, "src")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print(f"Device: {device}")


## Generate Training Data


In [ ]:
from data.generator import generate_transactions
from utils.seeds import set_global_seeds

set_global_seeds(42)
df = generate_transactions(n_txns=20_000, n_users=1_000, n_merchants=300, fraud_rate=0.03, seed=42)
print(f"Generated {len(df):,} transactions (fraud rate: {df['is_fraud'].mean():.3f})")


## Experiment 1: Tabular XGBoost Baseline


In [ ]:
from models.train import train_pipeline

result_xgb = train_pipeline(df, model_type="xgb_tabular", split_strategy="temporal", seed=42, epochs=30)
print("=== XGBoost Tabular ===")
for k, v in sorted(result_xgb["raw_metrics"].items()):
    print(f"  {k}: {v:.4f}")


## Experiment 2: GraphSAGE Only


In [ ]:
result_gs = train_pipeline(df, model_type="graphsage_only", split_strategy="temporal", seed=42, epochs=30)
print("=== GraphSAGE Only ===")
for k, v in sorted(result_gs["raw_metrics"].items()):
    print(f"  {k}: {v:.4f}")


## Experiment 3: GraphSAGE + XGBoost Hybrid (Flagship)


In [ ]:
result_hybrid = train_pipeline(df, model_type="graphsage_xgb", split_strategy="temporal", seed=42, epochs=30)
print("=== GraphSAGE + XGBoost (Hybrid) ===")
for k, v in sorted(result_hybrid["raw_metrics"].items()):
    print(f"  {k}: {v:.4f}")


## Experiment 4: GAT + XGBoost


In [ ]:
result_gat = train_pipeline(df, model_type="gat_xgb", split_strategy="temporal", seed=42, epochs=30)
print("=== GAT + XGBoost ===")
for k, v in sorted(result_gat["raw_metrics"].items()):
    print(f"  {k}: {v:.4f}")


## Side-by-Side Comparison


In [ ]:
from rich.table import Table
from rich.console import Console

console = Console()
table = Table(title="Model Ablation Results (Temporal Split)")
table.add_column("Model", style="cyan")
table.add_column("PR-AUC", style="green")
table.add_column("ROC-AUC", style="green")
table.add_column("Brier", style="yellow")
table.add_column("ECE", style="yellow")
table.add_column("Recall@1%FPR", style="magenta")

for name, result in [
    ("XGBoost Tabular", result_xgb),
    ("GraphSAGE Only", result_gs),
    ("GraphSAGE+XGBoost", result_hybrid),
    ("GAT+XGBoost", result_gat),
]:
    m = result["raw_metrics"]
    table.add_row(
        name,
        f"{m.get('raw_pr_auc', 0):.4f}",
        f"{m.get('raw_roc_auc', 0):.4f}",
        f"{m.get('raw_brier_score', 0):.4f}",
        f"{m.get('raw_ece', 0):.4f}",
        f"{m.get('raw_recall_at_1pct_fpr', 0):.4f}",
    )
console.print(table)


## Temporal Leakage Experiment

Compare random vs temporal splitting to demonstrate information leakage.


In [ ]:
result_random = train_pipeline(df, model_type="xgb_tabular", split_strategy="random", seed=42)
result_temporal = train_pipeline(df, model_type="xgb_tabular", split_strategy="temporal", seed=42)

print("=== Split Strategy Comparison (XGBoost) ===")
print(f"Random split  PR-AUC: {result_random['raw_metrics']['raw_pr_auc']:.4f}")
print(f"Temporal split PR-AUC: {result_temporal['raw_metrics']['raw_pr_auc']:.4f}")
print(f"\nDifference: {result_random['raw_metrics']['raw_pr_auc'] - result_temporal['raw_metrics']['raw_pr_auc']:.4f}")
print("(Positive difference confirms temporal leakage in random splits)")
